# 추가 백엔드-1 (Vision PASS/FAIL 집계) — Jupyter Notebook

이 노트북은 첨부된 설계 사양(2026-01-29 추가 백엔드-1 설계)에 따라,
- `Vision1~2` 로그 테이블에서 **주/야간 구간 필터링(KST 로컬 기준)**,
- `goodorbad='GoodFile'`만 사용,
- `barcode_information` **바코드별 최신 1건(dedup)**만 남긴 뒤,
- `result`의 `PASS/FAIL`을 **시간대(A~F 또는 A'~F')** 및 **품번(pn)** 기준으로 집계하여
출력 테이블(피벗)을 생성합니다.

> ⚠️ 주의: 이 노트북은 DB에 직접 접속합니다. 사내망/VPN/방화벽 설정을 확인하세요.


## 0) 실행 파라미터 설정

- `PROD_DAY`: `YYYYMMDD` (예: `20260127`)
- `SHIFT_TYPE`: `"day"` 또는 `"night"`


In [1]:
# ====== USER PARAMETERS ======
PROD_DAY = "20260127"      # YYYYMMDD
SHIFT_TYPE = "day"         # "day" or "night"
STATIONS = ["Vision1", "Vision2"]

# ====== DB CONFIG (Spec) ======
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

VISION_SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
VISION_TABLE  = "fct_vision_testlog_txt_processing_history"

REMARK_SCHEMA = "g_production_film"
REMARK_TABLE  = "remark_info"


## 1) 라이브러리 & 유틸

In [2]:
import re
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import List, Tuple, Optional

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine


In [3]:
def make_engine(cfg: dict) -> Engine:
    # psycopg2 드라이버 사용 (환경에 따라 psycopg2-binary 필요)
    url = f"postgresql+psycopg2://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    # 노트북 용도라 pool은 기본값 사용 (필요 시 조정)
    return create_engine(url, pool_pre_ping=True)


def parse_prod_day(prod_day: str) -> datetime:
    # prod_day: YYYYMMDD
    return datetime.strptime(prod_day, "%Y%m%d")


def shift_window(prod_day: str, shift_type: str) -> Tuple[datetime, datetime]:
    '''
    Spec (KST 로컬 기준):
      day   : [D] 08:30:00 ~ 20:29:59 (inclusive)
      night : [D] 20:30:00 ~ 23:59:59 + [D+1] 00:00:00 ~ 08:29:59 (inclusive)
    '''
    d0 = parse_prod_day(prod_day)
    if shift_type not in ("day", "night"):
        raise ValueError("SHIFT_TYPE must be 'day' or 'night'")

    if shift_type == "day":
        start = d0.replace(hour=8, minute=30, second=0)
        end   = d0.replace(hour=20, minute=29, second=59)
    else:
        start = d0.replace(hour=20, minute=30, second=0)
        end   = (d0 + timedelta(days=1)).replace(hour=8, minute=29, second=59)

    return start, end


@dataclass(frozen=True)
class Band:
    label: str
    start: datetime
    end: datetime


def make_bands(prod_day: str, shift_type: str) -> List[Band]:
    d0 = parse_prod_day(prod_day)
    d1 = d0 + timedelta(days=1)

    if shift_type == "day":
        return [
            Band("A",  d0.replace(hour=8,  minute=30, second=0), d0.replace(hour=10, minute=29, second=59)),
            Band("B",  d0.replace(hour=10, minute=30, second=0), d0.replace(hour=12, minute=29, second=59)),
            Band("C",  d0.replace(hour=12, minute=30, second=0), d0.replace(hour=14, minute=29, second=59)),
            Band("D",  d0.replace(hour=14, minute=30, second=0), d0.replace(hour=16, minute=29, second=59)),
            Band("E",  d0.replace(hour=16, minute=30, second=0), d0.replace(hour=18, minute=29, second=59)),
            Band("F",  d0.replace(hour=18, minute=30, second=0), d0.replace(hour=20, minute=29, second=59)),
        ]
    else:
        return [
            Band("A'", d0.replace(hour=20, minute=30, second=0), d0.replace(hour=22, minute=29, second=59)),
            Band("B'", d0.replace(hour=22, minute=30, second=0), d1.replace(hour=0,  minute=29, second=59)),
            Band("C'", d1.replace(hour=0,  minute=30, second=0), d1.replace(hour=2,  minute=29, second=59)),
            Band("D'", d1.replace(hour=2,  minute=30, second=0), d1.replace(hour=4,  minute=29, second=59)),
            Band("E'", d1.replace(hour=4,  minute=30, second=0), d1.replace(hour=6,  minute=29, second=59)),
            Band("F'", d1.replace(hour=6,  minute=30, second=0), d1.replace(hour=8,  minute=29, second=59)),
        ]


def normalize_time_text(s: Optional[str]) -> Optional[str]:
    '''
    DB end_time이 text이며, 'HH:MI:SS'로 정규화 요구.
    - '11:28:23.123' 같은 경우 '.fff' 제거
    - None/비정상 포맷은 None 반환
    '''
    if s is None:
        return None
    s = str(s).strip()
    if not s:
        return None
    s = re.sub(r"\..*$", "", s)  # drop fractional part
    if re.fullmatch(r"\d{2}:\d{2}:\d{2}", s):
        return s
    for fmt in ("%H:%M:%S", "%H:%M"):
        try:
            dt = datetime.strptime(s, fmt)
            return dt.strftime("%H:%M:%S")
        except Exception:
            pass
    return None


def barcode_key18(barcode: Optional[str]) -> Optional[str]:
    '''
    Spec: barcode_information의 18번째 문자(1-indexed)가 key.
    Python index: 17
    '''
    if barcode is None:
        return None
    b = str(barcode)
    return b[17] if len(b) >= 18 else None


## 2) DB에서 데이터 로드

- Vision 로그: `a1_fct_vision_testlog_txt_processing_history.fct_vision_testlog_txt_processing_history`
  - station in (Vision1, Vision2)
  - goodorbad = 'GoodFile'
  - shift window 조건에 해당하는 end_ts만
  - barcode_information별 최신 1건(row_number)만 유지
- 품번 매핑: `g_production_film.remark_info`
  - barcode_information(키 1글자) → pn, remark


In [4]:
engine = make_engine(DB_CONFIG)

start_ts, end_ts = shift_window(PROD_DAY, SHIFT_TYPE)
print("[WINDOW]", start_ts, "->", end_ts)

# 1) remark_info mapping load (key_char -> pn, remark)
remark_sql = text(f"""
SELECT
  barcode_information AS key_char,
  pn::text AS pn,
  remark::text AS remark
FROM {REMARK_SCHEMA}.{REMARK_TABLE}
""")
remark_df = pd.read_sql(remark_sql, engine)
remark_df["key_char"] = remark_df["key_char"].astype(str).str.strip()
print("[remark_info rows]", len(remark_df))

# 2) Vision data (dedup latest per barcode_information)
vision_sql = text(f"""
WITH base AS (
  SELECT
    station,
    end_day,
    regexp_replace(end_time, '\\..*$', '') AS end_time_norm,
    barcode_information,
    goodorbad,
    result,
    to_timestamp(end_day || ' ' || regexp_replace(end_time, '\\..*$', ''), 'YYYYMMDD HH24:MI:SS') AS end_ts
  FROM {VISION_SCHEMA}.{VISION_TABLE}
  WHERE station = ANY(:stations)
    AND goodorbad = 'GoodFile'
),
filt AS (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY barcode_information
      ORDER BY end_ts DESC
    ) AS rn
  FROM base
  WHERE end_ts >= :start_ts
    AND end_ts <= :end_ts
)
SELECT
  station, end_day, end_time_norm, end_ts,
  barcode_information, result
FROM filt
WHERE rn = 1
""")

df = pd.read_sql(
    vision_sql,
    engine,
    params={
        "stations": STATIONS,
        "start_ts": start_ts,
        "end_ts": end_ts,
    },
)

print("[vision rows after window+dedup]", len(df))
df.head(5)


[WINDOW] 2026-01-27 08:30:00 -> 2026-01-27 20:29:59
[remark_info rows] 6
[vision rows after window+dedup] 3902


,station,end_day,end_time_norm,end_ts,barcode_information,result
0,Vision2,20260127,19:50:49,2026-01-27 10:50:49+00:00,BA1WJ26027500751PC3T-14F014-AC,PASS
1,Vision2,20260127,19:50:34,2026-01-27 10:50:34+00:00,BA1WJ26027500752PC3T-14F014-AC,PASS
2,Vision2,20260127,19:50:16,2026-01-27 10:50:16+00:00,BA1WJ26027500753PC3T-14F014-AC,PASS
3,Vision2,20260127,19:50:03,2026-01-27 10:50:03+00:00,BA1WJ26027500754PC3T-14F014-AC,PASS
4,Vision2,20260127,19:49:47,2026-01-27 10:49:47+00:00,BA1WJ26027500755PC3T-14F014-AC,PASS


## 3) 품번(pn) 매핑 & 시간대(Band) 분류

In [7]:
# Normalize time text again in Python (safety)
df["end_time_norm"] = df["end_time_norm"].apply(normalize_time_text)

# Extract 18th char key and map to pn/remark
df["key_char"] = df["barcode_information"].apply(barcode_key18)

# merge (left join)
df = df.merge(remark_df, on="key_char", how="left")

# =========================
# [FIX-1] pn 컬럼 보정 (merge 결과가 pn / pn_x / pn_y / PN 등 어떤 형태든 대응)
# =========================
def _pick_first_existing_col(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    return None

# 1) 표준 후보들에서 pn 컬럼 찾기
pn_col = _pick_first_existing_col(df, ["pn", "PN", "Pn", "p_n", "product_number", "pn_y", "pn_x"])

if pn_col is None:
    # 아예 없으면 생성
    df["pn"] = "UNKNOWN"
else:
    # pn_col을 표준 pn으로 복사
    df["pn"] = df[pn_col]

# (선택) pn_x/pn_y 둘 다 있는 경우: pn_y(remark에서 온 값) 우선, 없으면 pn_x
if "pn_y" in df.columns and "pn_x" in df.columns:
    df["pn"] = df["pn_y"].fillna(df["pn_x"])

# 결측/공백 처리
df["pn"] = (
    df["pn"]
      .astype(str)
      .str.strip()
      .replace({"": "UNKNOWN", "nan": "UNKNOWN", "None": "UNKNOWN"})
      .fillna("UNKNOWN")
)

# =========================
# [FIX-2] end_ts timezone 통일 (tz-aware/naive 비교 에러 방지)
# =========================
df["end_ts"] = pd.to_datetime(df["end_ts"], errors="coerce")

# tz-aware면 KST로 맞춘 뒤 tz 제거 -> naive 비교
if hasattr(df["end_ts"].dt, "tz") and df["end_ts"].dt.tz is not None:
    df["end_ts"] = df["end_ts"].dt.tz_convert("Asia/Seoul").dt.tz_localize(None)

# Band assignment based on end_ts and shift bands
bands = make_bands(PROD_DAY, SHIFT_TYPE)

def assign_band(ts):
    if pd.isna(ts):
        return None
    t = ts.to_pydatetime() if hasattr(ts, "to_pydatetime") else pd.to_datetime(ts).to_pydatetime()
    for b in bands:
        if b.start <= t <= b.end:
            return b.label
    return None

df["band"] = df["end_ts"].apply(assign_band)
df = df.dropna(subset=["band"])

print("[rows after band assignment]", len(df))

# 출력 컬럼이 없을 수도 있으니 안전하게 존재하는 것만 표시
cols = ["station","end_ts","band","barcode_information","key_char","pn","remark","result"]
cols = [c for c in cols if c in df.columns]
df[cols].head(10)

[rows after band assignment] 3902


,station,end_ts,band,barcode_information,key_char,pn,remark,result
0,Vision2,2026-01-27 19:50:49,F,BA1WJ26027500751PC3T-14F014-AC,C,35643009,Non-PD,PASS
1,Vision2,2026-01-27 19:50:34,F,BA1WJ26027500752PC3T-14F014-AC,C,35643009,Non-PD,PASS
2,Vision2,2026-01-27 19:50:16,F,BA1WJ26027500753PC3T-14F014-AC,C,35643009,Non-PD,PASS
3,Vision2,2026-01-27 19:50:03,F,BA1WJ26027500754PC3T-14F014-AC,C,35643009,Non-PD,PASS
4,Vision2,2026-01-27 19:49:47,F,BA1WJ26027500755PC3T-14F014-AC,C,35643009,Non-PD,PASS
5,Vision2,2026-01-27 19:49:32,F,BA1WJ26027500756PC3T-14F014-AC,C,35643009,Non-PD,PASS
6,Vision2,2026-01-27 19:49:18,F,BA1WJ26027500757PC3T-14F014-AC,C,35643009,Non-PD,PASS
7,Vision2,2026-01-27 19:53:08,F,BA1WJ26027500758PC3T-14F014-AC,C,35643009,Non-PD,PASS
8,Vision2,2026-01-27 19:52:53,F,BA1WJ26027500759PC3T-14F014-AC,C,35643009,Non-PD,PASS
9,Vision2,2026-01-27 19:52:40,F,BA1WJ26027500760PC3T-14F014-AC,C,35643009,Non-PD,PASS


## 4) 집계 (pn × Band × PASS/FAIL) → 피벗 출력

In [14]:
# =========================
# 4) 집계 (pn × Band × PASS/FAIL) → 피벗 출력
# =========================

from datetime import datetime  # ✅ 추가

# Base grouping
g = (
    df.groupby(["pn", "band", "result"], dropna=False)
      .size()
      .rename("cnt")
      .reset_index()
)

# Ensure PASS/FAIL columns
pivot_pf = (
    g.pivot_table(
        index=["pn", "band"],
        columns="result",
        values="cnt",
        aggfunc="sum",
        fill_value=0
      )
      .reset_index()
)

for col in ["PASS", "FAIL"]:
    if col not in pivot_pf.columns:
        pivot_pf[col] = 0

# Make display string
pivot_pf["cell"] = pivot_pf.apply(
    lambda r: f"PASS: {int(r.get('PASS', 0))}, FAIL: {int(r.get('FAIL', 0))}",
    axis=1
)

band_order = [b.label for b in bands]

final = (
    pivot_pf.pivot_table(index=["pn"], columns="band", values="cell", aggfunc="first")
            .reindex(columns=band_order)
            .reset_index()
)

final.insert(0, "shift_type", SHIFT_TYPE)
final.insert(0, "prod_day", PROD_DAY)

# ✅ 화면 상단에 보이는 'band' 제거 (컬럼축 이름 제거)
final.columns.name = None

# ✅ 컬럼명 한글 라벨로 변경
band_label_map = {
    "A":  "A시간대(08:30:00 ~ 10:29:59)",
    "B":  "B시간대(10:30:00 ~ 12:29:59)",
    "C":  "C시간대(12:30:00 ~ 14:29:59)",
    "D":  "D시간대(14:30:00 ~ 16:29:59)",
    "E":  "E시간대(16:30:00 ~ 18:29:59)",
    "F":  "F시간대(18:30:00 ~ 20:29:59)",
    "A'": "A'시간대(20:30:00 ~ 22:29:59)",
    "B'": "B'시간대(22:30:00 ~ 00:29:59)",
    "C'": "C'시간대(00:30:00 ~ 02:29:59)",
    "D'": "D'시간대(02:30:00 ~ 04:29:59)",
    "E'": "E'시간대(04:30:00 ~ 06:29:59)",
    "F'": "F'시간대(06:30:00 ~ 08:29:59)",
}
final = final.rename(columns=band_label_map)

# ✅ final 생성/저장 시각 컬럼 추가 (맨 오른쪽)
final["df_updated_at"] = datetime.now()

final

,prod_day,shift_type,pn,A시간대(08:30:00 ~ 10:29:59),B시간대(10:30:00 ~ 12:29:59),C시간대(12:30:00 ~ 14:29:59),D시간대(14:30:00 ~ 16:29:59),E시간대(16:30:00 ~ 18:29:59),F시간대(18:30:00 ~ 20:29:59),df_updated_at
0,20260127,day,35643009,"PASS: 674, FAIL: 0","PASS: 739, FAIL: 0","PASS: 727, FAIL: 0","PASS: 446, FAIL: 0","PASS: 744, FAIL: 0","PASS: 570, FAIL: 2",2026-01-29 16:17:34.485799
